# Moteur de recherche — Partie 3

Interface du moteur de recherche. Conformément aux consignes, **seule l'interface vit dans ce notebook** :
le chargement du corpus, l'indexation et la recherche sont réalisés par les modules `.py` de `source/backend`.

*Author: Corentin Legrand*

## 1. Préparation : réutiliser le moteur

### 1.1 Import des classes et chargement du corpus

Les modules du backend sont rendus importables en ajoutant `source/backend` au chemin de recherche de Python.
Le notebook doit être exécuté depuis son dossier (`source/frontend`), comportement par défaut de VS Code et Jupyter.

In [ ]:
import sys
from pathlib import Path

BACKEND_PATH: Path = Path.cwd().parent / "backend"
if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(0, str(BACKEND_PATH))

In [ ]:
from helpers.environment_manager import EnvironmentManager
from infrastructure.json_dao import JsonDAO
from infrastructure.json_reader import JsonReader
from infrastructure.json_writer import JsonWriter
from models.corpus import Corpus
from models.document import Document
from script.scrap_data import DATA_FILENAME_KEY
from service.author_manager import AuthorManager
from service.corpus_manager import CorpusManager
from service.corpus_mapper import CorpusMapper
from service.document_factory import DocumentFactory
from service.search_engine import SearchEngine

Le corpus est rechargé depuis le fichier produit en Partie 1 (`load`) : les API Reddit et Arxiv ne sont pas
réinterrogées.

In [ ]:
data_filename: str = EnvironmentManager.get_instance().get_environment_variable(DATA_FILENAME_KEY)
mapper: CorpusMapper = CorpusMapper(DocumentFactory(), AuthorManager())
corpus_manager: CorpusManager = CorpusManager(JsonDAO(JsonWriter(), JsonReader()), mapper, data_filename)
corpus_manager.load()

corpus: Corpus = Corpus.get_instance()
first_document: Document = next(iter(corpus.documents.values()))
print(repr(corpus_manager))
print(f"First document: {first_document}")

### 1.2 et 1.3 Instanciation du moteur

La construction des matrices TF et TF×IDF affiche une barre de progression `tqdm` sur la boucle des documents
(`TermIndexBuilder`). La recherche elle-même est un produit matriciel, sans boucle Python à suivre.

In [ ]:
search_engine: SearchEngine = SearchEngine(corpus)

Vérification rapide par quelques requêtes en mots-clés (rappel de la Partie 2).

In [ ]:
from IPython.display import display

NB_RESULTS: int = 5
TEST_QUERIES: list[str] = ["climate change", "quantum error correction", "solar energy"]

for query in TEST_QUERIES:
    print(f"Query: {query}")
    display(search_engine.search(query, NB_RESULTS))